# Предобработка данных «СберАвтоподписки»

Исходные `ga_sessions.csv` и `ga_hits.csv` весят около 4.5 ГБ, поэтому в репозитории их нет. Ноутбук оставлен с сохранёнными выводами, результат его работы лежит в `../data/data.csv`

Целевое действие: любое из событий `sub_*_click` / `sub_submit_success`. Сессии и хиты склеены по `session_id`, после чего признаки кодируются в числа, а словари кодировок сохраняются в `../encoders/`

In [59]:
# Импорт библиотек
import pandas as pd
import pickle

In [60]:
#чтение файла ga_sessions.csv
ga_sessions_df = pd.read_csv("../data/ga_sessions.csv")
ga_sessions_df.shape

(1860042, 18)

In [61]:
#чтение файла ga_hits.csv
ga_hits_df = pd.read_csv("../data/ga_hits.csv")
ga_hits_df.shape

(15726470, 11)

In [62]:
#объединение датафреймов
data_df = pd.merge(ga_sessions_df, ga_hits_df, on = "session_id")
data_df.shape

(15685219, 28)

In [63]:
#пропуски в данных
missing_values = data_df.isnull().sum()
print(missing_values)

session_id                         0
client_id                          0
visit_date                         0
visit_time                         0
visit_number                       0
utm_source                       700
utm_medium                         0
utm_campaign                 2198873
utm_adcontent                2832418
utm_keyword                  9204827
device_category                    0
device_os                    9158382
device_brand                 3945875
device_model                15562932
device_screen_resolution           0
device_browser                     0
geo_country                        0
geo_city                           0
hit_date                           0
hit_time                     9160203
hit_number                         0
hit_type                           0
hit_referer                  6235498
hit_page_path                      0
event_category                     0
event_action                       0
event_label                  3727682
e

In [64]:
#удаление ненужных столбцов
data_df.drop(["event_value", "device_model", "utm_keyword", "device_os", "hit_time", 
            "hit_referer", "session_id", "client_id", "hit_date", "hit_type", 
            "visit_number", "visit_date", "visit_time", 'hit_number', 
            "hit_page_path", "event_category", "event_label"], axis= 1 , inplace = True )

In [65]:
#поиск и удаление дубликатов
duplicates = data_df.duplicated()
print("Количество дубликатов:", duplicates.sum())
data_df = data_df.drop_duplicates()

Количество дубликатов: 13779729


In [66]:
#пропуски в данных
missing_values = data_df.isnull().sum()
print(missing_values)

utm_source                     187
utm_medium                       0
utm_campaign                170524
utm_adcontent               280318
device_category                  0
device_brand                456600
device_screen_resolution         0
device_browser                   0
geo_country                      0
geo_city                         0
event_action                     0
dtype: int64


In [67]:
#замена целевой переменной на 1 и 0, вывод их количества
data_df["event_action"] = data_df["event_action"].replace('sub_car_claim_click', 1)
data_df["event_action"] = data_df["event_action"].replace('sub_car_claim_submit_click', 1)
data_df["event_action"] = data_df["event_action"].replace('sub_open_dialog_click', 1)
data_df["event_action"] = data_df["event_action"].replace('sub_custom_question_submit_click', 1)
data_df["event_action"] = data_df["event_action"].replace('sub_call_number_click', 1)
data_df["event_action"] = data_df["event_action"].replace('sub_callback_submit_click', 1)
data_df["event_action"] = data_df["event_action"].replace('sub_submit_success', 1)
data_df["event_action"] = data_df["event_action"].replace('sub_car_request_submit_click', 1)

data_df["event_action"] = data_df["event_action"].apply(lambda x: x if x == 1 else 0)

data_df["event_action"].value_counts()

event_action
0    1860620
1      44870
Name: count, dtype: int64

In [68]:
#перевод столбца device_screen_resolution в формат площади экрана
data_df.device_screen_resolution = data_df.device_screen_resolution.apply(lambda x: int(x.split("x")[0]) * int(x.split("x")[1]))

In [69]:
#поиск и удаление дубликатов
duplicates = data_df.duplicated()
print("Количество дубликатов:", duplicates.sum())
data_df = data_df.drop_duplicates()

Количество дубликатов: 1579149


In [70]:
# Сохраняем словари с кодировками словесной информации в числовую в папку encoders 
path = '../encoders/' 
cols_to_encode = [ 
    'utm_source', 
    'utm_medium', 
    'utm_campaign', 
    'utm_adcontent', 
    'device_category', 
    'device_brand', 
    'device_browser', 
    'geo_country', 
    'geo_city'] 
 
for col in cols_to_encode: 
    d = {v:i for i,v in enumerate(data_df[col].astype('category').cat.categories)} 
    with open(f'{path}{col}_encoder.pickle', 'wb') as f: 
        pickle.dump(d, f)


In [71]:
#замена значений всех столбцов на числовые
for col in ["utm_medium", "utm_source", "utm_campaign", "utm_adcontent", "device_category", 
           "device_brand", "device_browser", "geo_country", "geo_city"]: 
    data_df[col] = data_df[col].astype("category").cat.codes

In [72]:
#финальный результат целевой переменной
data_df["event_action"].value_counts()

event_action
0    305151
1     21190
Name: count, dtype: int64

In [73]:
#проверка корреляции
for col in data_df.columns[2:]: 
    print(data_df[[col, "event_action"]].corr(method = "spearman")) 
    print()

              utm_campaign  event_action
utm_campaign      1.000000     -0.023898
event_action     -0.023898      1.000000

               utm_adcontent  event_action
utm_adcontent       1.000000     -0.028558
event_action       -0.028558      1.000000

                 device_category  event_action
device_category         1.000000     -0.010892
event_action           -0.010892      1.000000

              device_brand  event_action
device_brand      1.000000     -0.010106
event_action     -0.010106      1.000000

                          device_screen_resolution  event_action
device_screen_resolution                  1.000000      0.013588
event_action                              0.013588      1.000000

                device_browser  event_action
device_browser        1.000000      0.009613
event_action          0.009613      1.000000

              geo_country  event_action
geo_country      1.000000     -0.003814
event_action    -0.003814      1.000000

              geo_city  eve

In [74]:
#размер конечного датафрейма
data_df.shape

(326341, 11)

In [75]:
#типы данных в конечном датафрейме
data_df.dtypes

utm_source                  int16
utm_medium                   int8
utm_campaign                int16
utm_adcontent               int16
device_category              int8
device_brand                int16
device_screen_resolution    int64
device_browser               int8
geo_country                 int16
geo_city                    int16
event_action                int64
dtype: object

In [76]:
#конечный датафрейм
data_df.head(10)

,utm_source,utm_medium,utm_campaign,utm_adcontent,device_category,device_brand,device_screen_resolution,device_browser,geo_country,geo_city,event_action
0,149,6,84,244,1,76,259200,6,117,2374,0
2,77,15,39,263,1,145,328790,46,117,1346,0
3,149,6,84,244,1,76,259200,6,117,1047,0
19,210,14,-1,68,1,191,308898,6,117,1346,0
22,210,14,-1,-1,1,191,308898,6,117,1346,0
24,210,31,87,45,1,10,304500,44,117,1811,0
83,116,14,39,57,2,103,619458,50,117,1811,0
89,149,6,84,45,1,145,230400,6,117,1811,0
98,149,6,84,244,1,10,329160,44,117,1346,0
108,149,6,84,45,1,145,376980,6,117,1346,0


In [77]:
#сохранение в файл csv
data_df.to_csv("../data/data.csv", index=False)